In [1]:
from getpass import getpass
import os

token = getpass("GitHub Token: ")
os.environ["GITHUB_TOKEN"] = token


GitHub Token: ··········


In [2]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://${GITHUB_TOKEN}@github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing

/content
Cloning into 'ddpm_option_pricing'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 41 (delta 20), reused 28 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 10.30 KiB | 46.00 KiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/ddpm_option_pricing


In [3]:
import torch
import numpy as np
import math

from src.schedules import make_alpha_schedule
from src.ddpm_model import ScoreMLP
from src.train_ddpm import train_ddpm
from src.sample_ddpm import sample_returns_Q_std
from src.price_options import price_vanilla_with_ddpm


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S0 = 100.0
mu = 0.08
sigma = 0.20
r = 0.04
days_per_year = 252
dt = 1.0 / days_per_year

N_train = 1_000
T = 1000  # diffusion steps

v0 = sigma**2 * dt
s0 = math.sqrt(v0)
mP = (mu - 0.5 * sigma**2) * dt

rng = np.random.default_rng(42)
torch.manual_seed(42)
np.random.seed(42)

# THIS is the important part:
y_train = rng.normal(loc=mP, scale=s0, size=(N_train,)).astype(np.float32)
y_train_std = (y_train - mP) / s0



In [5]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(torch.from_numpy(y_train_std).unsqueeze(1)),
                          batch_size=512, shuffle=True)


In [6]:
betas, alphas, alphas_bar = make_alpha_schedule(T, device)


In [7]:
model = ScoreMLP(hidden_dim=256, time_emb_dim=64).to(device)

model = train_ddpm(
    model=model,
    train_loader=train_loader,
    alphas_bar=alphas_bar,
    T=T,
    device=device,
    epochs=60
)


Epoch 1/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 2/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 3/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 4/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 5/60:   0%|          | 0/2 [00:00<?, ?it/s]

...

Epoch 56/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 57/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 58/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 59/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 60/60:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
@torch.no_grad()
def p_check_bias_scale(model, n=1000):
    model.eval()
    B = n
    y = torch.randn(B, 1, device=device)
    for t in reversed(range(T)):
        a_bar_t = alphas_bar[t]
        t01 = torch.full((B,), (t + 0.5)/T, device=device)
        eps_hat = model(y, t01)
        if t > 0:
            post_var = (1 - alphas_bar[t-1]) / (1 - a_bar_t) * betas[t]
            post_var = torch.clamp(post_var, min=1e-12)
            mean = (1 / torch.sqrt(alphas[t])) * (
                y - (betas[t] / torch.sqrt(1 - a_bar_t)) * eps_hat
            )
            y = mean + torch.sqrt(post_var) * torch.randn_like(y)
        else:
            y = (1 / torch.sqrt(alphas[t])) * (
                y - (betas[t] / torch.sqrt(1 - a_bar_t)) * eps_hat
            )
    y_std = y.view(-1).cpu().numpy()
    return float(y_std.mean()), float(y_std.std(ddof=1))

m_hat, s_hat = p_check_bias_scale(model)
print(m_hat, s_hat)


0.03980404511094093 0.9734224677085876


In [9]:
betas, alphas, alphas_bar = make_alpha_schedule(T, device)

result = price_vanilla_with_ddpm(
    model=model,
    S0=S0,
    K=100.0,
    r=r,
    sigma=sigma,
    mu=mu,
    H_steps=252,
    n_paths=1000,
    alphas=alphas,
    alphas_bar=alphas_bar,
    betas=betas,
    m_hat_P=m_hat,
    s_hat_P=s_hat,
    dt=dt,
    device=device,
    use_ddim=False,
)
print(result)

{'P_m_hat': 0.03980404511094093, 'P_s_hat': 0.9734224677085876, 'returns_mean': 7.93651634012349e-05, 'returns_std': 0.012319937348365784, 'martingale': {21: np.float64(100.35200176283416), 63: np.float64(99.86904127157969), 126: np.float64(100.04212328397975), 252: np.float64(99.87675715808312)}, 'ks_stat': 0.019401721538323702, 'ks_pvalue': 0.838587724912467, 'ddpm_price': np.float64(9.601359667026683), 'ddpm_stderr': np.float64(0.4433440873266194), 'bs_price': np.float64(9.925053717274437), 'abs_diff': 0.3236940502477541}
